# 01 — connect to the lake

DuckDB attached to Lakekeeper, reading MinIO on the published ports. Every table the lake has, with its row count, in one query per namespace.

In [ ]:
from k2lake import connect
con = connect()
con.sql("SELECT table_schema AS namespace, table_name FROM information_schema.tables WHERE table_catalog = 'lake' ORDER BY 1, 2").show(max_rows=60)

Row counts come from the Iceberg metadata (no scan), so this is cheap even for `raw.messages`.

In [ ]:
tables = con.sql("SELECT table_schema, table_name FROM information_schema.tables WHERE table_catalog = 'lake' ORDER BY 1, 2").fetchall()
for ns, t in tables:
    n = con.sql(f'SELECT count(*) FROM lake.{ns}.{t}').fetchone()[0]
    print(f'{ns}.{t:<28} {n:>14,}')

The one thing to verify before trusting any timestamp below: the session is UTC.

In [ ]:
con.sql("SELECT current_setting('TimeZone') AS tz, min(exchange_ts) AS first_trade, max(exchange_ts) AS last_trade FROM lake.gold.trades").show()